In [1]:
def _extract_vision(pdf_path) -> list[dict]:
    """Adapter vision → format benchmarku. Używa CACHE, nie API.

    Skip jeśli jakikolwiek rozdział I–X nie ma kompletnego cache.
    """
    from backend.app.config import VISION_CACHE_DIR
    from backend.app.document.extraction_cache import (
        chapter_cache_path,
        chapter_is_complete,
        load_cached_chapters,
    )
    from backend.app.document.structure_extractor import extract_structure

    document, doc = extract_structure(pdf_path)
    doc.close()
    for ch in document.chapters:
        if not chapter_cache_path(VISION_CACHE_DIR, ch.chapter_id).exists():
            pytest.skip(f"Brak cache dla rozdziału {ch.chapter_id}")
    load_cached_chapters(document, VISION_CACHE_DIR)
    if not all(chapter_is_complete(ch) for ch in document.chapters):
        pytest.skip("Cache niekompletny")

    return [
        {"text": " ".join(b.text for b in p.blocks), "page": p.page_num}
        for p in document.get_all_pages()
    ]


In [2]:
from pathlib import Path

PDF_PATH = Path("../data/raw/raport_2024_pl.pdf")



In [3]:
import sys
sys.path.insert(0, str(Path("..").resolve()))

result_vison = _extract_vision(PDF_PATH)

In [4]:
print(result_vison)

[{'text': '', 'page': 3}, {'text': '1. List Prezesa Zarządu GRI 2-22 Zdjęcie portretowe mężczyzny w garniturze i okularach, ujęcie w okrągłej ramce. Szanowni Państwo, W minionym roku obchodziliśmy jubileusz stulecia BGK. To wielki przywilej kierować instytucją z tak wielkim dziedzictwem i znaczeniem dla rozwoju społeczno-gospodarczego kraju. Warto podkreślić jednak, że choć misja polskiego banku rozwoju pozostaje niezmieniona, w zależności od uwarunkowań gospodarczych podejmujemy działania, które w największy sposób odpowiadają na aktualne potrzeby. Rok 2024 przyniósł wzrost PKB Polski o 2,9% r/r. Nasza gospodarka radzi sobie dobrze, zwłaszcza na tle innych państw UE. Jesteśmy jednak świadomi, że przed Europą stoją poważne wyzwania - utrzymanie konkurencyjności i rozwój innowacji. Sytuacja geopolityczna nie pozostawia złudzeń, że priorytetem Polski musi być wzmacnianie odporności na zmiany i bezpieczeństwo strategiczne. Rozumieje je zarówno w kontekście militarnym, jak i inwestycji w s

In [5]:
from backend.app.document.vision_extractor import load_or_extract
document = load_or_extract(PDF_PATH)

In [6]:
ch =  document.get_chapter('I')
page_num = 4
section_header_on_page = [s.text.split(maxsplit=1)[1] for s in ch.get_blocks_by_type("section-header") if s.page == page_num]
print(section_header_on_page)

['List Prezesa Zarządu']


# WAZNE - do testow i do posporcesu?? i zupdatować extrakcje??

In [7]:
for ch in document.chapters:
    for page in ch.pages:
        section_header_on_page = [s.text.split(maxsplit=1)[1] for s in ch.get_blocks_by_type("section-header") if s.page == page.page_num]
        if section_header_on_page != page.sections:
            print(page.page_num)
            print(page.sections)
            print(section_header_on_page)
            
# WAZNE


4
['1. List Prezesa Zarządu']
['List Prezesa Zarządu']
6
['2. Kim jesteśmy?', '3. Podsumowanie 2024 roku']
['Kim jesteśmy?', 'Podsumowanie 2024 roku']
9
['4. Najważniejsze wydarzenia w 2024 roku']
['Najważniejsze wydarzenia w 2024 roku']
10
['5. Charakterystyka Grupy Kapitałowej BGK']
['Charakterystyka Grupy Kapitałowej BGK']
11
['6. Kluczowe dane finansowe']
['Kluczowe dane finansowe']
15
['1. Strategia, czyli co nas wyróżnia', '2. Programy modelu biznesowego jako odpowiedź na potrzeby rynku']
['Strategia, czyli co nas wyróżnia', 'Programy modelu biznesowego jako odpowiedź na potrzeby rynku']
21
['3.  Model tworzenia wartości']
['Model tworzenia wartości']
22
['4.  Kapitały wykorzystywane przez BGK']
['Kapitały wykorzystywane przez BGK']
30
['1.  BGK jako pracodawca']
['BGK jako pracodawca']
31
['2.  Warunki pracy']
['Warunki pracy']
37
['3.  Dbałość o pracowników']
['Dbałość o pracowników']
43
['4.  Zarządzanie różnorodnością']
['Zarządzanie różnorodnością']
48
['1.  Otoczenie społec

In [8]:
document.get_page(128)

ExtractedPage(page_num=128, content_rect=BBox(x0=210, y0=0.0, x1=1920.0, y1=1080.0), chapter='VI Ład korporacyjny', sections=['7.  Zasady i zarządzanie ładem korporacyjnym, w tym w obszarze zrównoważonego rozwoju'], blocks=[ExtractedBlock(block_id='p128_b0', page=128, element_type='subsection-header', text='Raportowanie incydentów i zdarzeń krytycznych', bbox=BBox(x0=0, y0=0, x1=0, y1=0), heading_level=2), ExtractedBlock(block_id='p128_b1', page=128, element_type='identifier', text='GRI 2-16', bbox=BBox(x0=0, y0=0, x1=0, y1=0), heading_level=None), ExtractedBlock(block_id='p128_b2', page=128, element_type='text', text='Zdarzenia krytyczne są w BGK raportowane do Zarządu. w banku przygotowaliśmy własną klasyfikację zdarzeń krytycznych, uznając, że są to:', bbox=BBox(x0=0, y0=0, x1=0, y1=0), heading_level=None), ExtractedBlock(block_id='p128_b3', page=128, element_type='list', text='- incydenty bezpieczeństwa informacji na poziomie krytycznym,\n- incydenty uznane za poważne zgodnie z def

In [9]:
import backend.app.document.vision_extractor as ve
from backend.app.document.structure_extractor import extract_structure
document = ve.load_or_extract(PDF_PATH)

In [10]:
block_types = set([block.element_type for block in document.get_all_blocks()])

In [11]:
block_types

{'caption',
 'footnote',
 'identifier',
 'infographic',
 'list',
 'picture',
 'section-header',
 'subsection-header',
 'table',
 'text'}

In [12]:
for block_type in block_types:
    print('='*60)
    print(block_type)
    print('='*60)
    for block in document.get_blocks_by_type(block_type)[:30]:
        print(block)
    print("\n")


table
ExtractedBlock(block_id='p11_b11', page=11, element_type='table', text='Podstawowe parametry finansowe działalności Grupy Kapitałowej BGK w latach 2020–2024.\nWartości wyrażone w mln zł.\n\n| dochodowość | 2024 | 2023 | 2022 | 2021 | 2020 |\n|---|---|---|---|---|---|\n| Wynik na działalności bankowej | 5 319 | 5 265 | 3 704 | 1 565 | 1 459 |\n| Koszty administracyjne | -979 | -868 | -685 | -652 | -624 |\n| Wynik z tytułu rezerw/odpisów | -673 | -399 | -301 | -332 | -385 |\n| Udział w zyskach i stratach jednostek stowarzyszonych | 733 | 705 | 32 | 253 | -25 |\n| Wynik brutto | 4 361 | 4 561 | 2 673 | 1 094 | 447 |\n| Wynik netto | 3 562 | 3 732 | 2 162 | 875 | 367 |', bbox=BBox(x0=0, y0=0, x1=0, y1=0), heading_level=None)
ExtractedBlock(block_id='p12_b0', page=12, element_type='table', text='Wybrane pozycje bilansowe BGK w latach 2020–2024. Wartości wyrażone w milionach złotych.\n\n| wybrane pozycje bilansowe | 31.12.2024 | 31.12.2023 | 31.12.2022 | 31.12.2021 | 31.12.2020 |\n|---

In [13]:
document.get_page(137
                  ).blocks

[ExtractedBlock(block_id='p137_b0', page=137, element_type='section-header', text='4. Ryzyko operacyjne', bbox=BBox(x0=0, y0=0, x1=0, y1=0), heading_level=1),
 ExtractedBlock(block_id='p137_b1', page=137, element_type='subsection-header', text='Organizacja procesu zarządzania ryzykiem operacyjnym', bbox=BBox(x0=0, y0=0, x1=0, y1=0), heading_level=2),
 ExtractedBlock(block_id='p137_b2', page=137, element_type='text', text='Ryzykiem operacyjnym zarządzamy w całym BGK, fundacjach i podmiotach zależnych. Oceniamy m.in. produkty, procesy i systemy. Pod uwagę brane są m.in. struktura, specyfika działalności, użytkowane systemy informatyczne, specyfika klientów i ich skargi, kadry, zmiany organizacyjne oraz rotacja pracowników. Uwzględniamy też czynniki zewnętrzne. Ryzyko operacyjne jest regularnie raportowane wyższej kadrze kierowniczej, Komitetowi Ryzyka Operacyjnego i Kontroli Wewnętrznej, Zarządowi oraz Radzie Nadzorczej i Komitetowi ds. Ryzyka.', bbox=BBox(x0=0, y0=0, x1=0, y1=0), headin

In [14]:
document.get_page(83
                  ).blocks

[ExtractedBlock(block_id='p83_b0', page=83, element_type='section-header', text='5. Transformacja cyfrowa i procesowa', bbox=BBox(x0=0, y0=0, x1=0, y1=0), heading_level=1),
 ExtractedBlock(block_id='p83_b1', page=83, element_type='text', text='Współczesna rzeczywistość biznesowa jest kształtowana przez dynamiczny rozwój technologii i procesów. Aby skutecznie konkurować i osiągać sukces, przedsiębiorstwa muszą przyjmować holistyczne podejście, uwzględniając zarówno aspekty technologiczne, jak i społeczne. Takie podejście stanowi fundament Transformacji cyfrowej i procesowej.', bbox=BBox(x0=0, y0=0, x1=0, y1=0), heading_level=None),
 ExtractedBlock(block_id='p83_b2', page=83, element_type='text', text='Rozwój cyfrowy BGK to kompleksowa odpowiedź na aktualne trendy i wyzwania związane z cyfryzacją instytucji finansowych. Wdrażamy nowoczesne systemy integrujące kluczowe procesy i funkcje biznesowe, w tym kanały elektroniczne czy platformy płatnicze. Sukcesywnie redukujemy dług technologicz